# 09 — Entity-level translation fidelity

BLEU says predicted markup costs $-0.09$, which is compatible with two different
stories: markup is damaging entity rendering, or markup neither helps nor hurts
entities while shifting other words. This notebook separates them.

For every test sentence we extract the entities present in the **English
reference** with the same spaCy model used for projection, then measure what
fraction of them survive into each system's output. This is the question the
markup was supposed to address, and BLEU cannot answer it: an entity is a couple
of tokens out of twenty, so getting it wrong barely moves a corpus $n$-gram
score.

Requires spaCy in the `tka` environment:
`pip install spacy` then `python -m spacy download en_core_web_sm`.

**Run from the repository root.** About 5 minutes; no GPU needed.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, re, sys, unicodedata
from collections import Counter, defaultdict
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

MT_DATA = ROOT / "data" / "processed" / "mt_v2"
RES     = ROOT / "results" / "mt"
FIGS    = ROOT / "paper" / "figures"

CONDS = ["baseline", "marked", "predicted", "predicted_raw"]
LABEL = {"baseline": "no markup", "marked": "oracle (projection)",
         "predicted": "predicted, stem-aligned", "predicted_raw": "predicted, whole-token"}

for c in CONDS:
    p = RES / f"hyps_{c}_v2.json"
    print(("  ok   " if p.exists() else "  MISS ") + p.name)
    if not p.exists():
        sys.exit("Run notebooks 07 and 08 first")

def write_json(obj, path, **kw):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, **kw)

Repo root: C:\Users\Haulai\mizo-ner
  ok   hyps_baseline_v2.json
  ok   hyps_marked_v2.json
  ok   hyps_predicted_v2.json
  ok   hyps_predicted_raw_v2.json


## Cell 2: Load references and hypotheses

In [3]:
refs = open(MT_DATA / "test_english.txt", encoding="utf-8").read().splitlines()
N = len(refs)
hyps = {c: json.load(open(RES / f"hyps_{c}_v2.json", encoding="utf-8")) for c in CONDS}
for c in CONDS:
    assert len(hyps[c]) == N, f"{c}: {len(hyps[c])} != {N}"
print(f"{N:,} test sentences, {len(CONDS)} conditions")

# the NER-unseen subset, recomputed here so this notebook stands alone
pred = json.load(open(RES / "predicted_tag_v2.json", encoding="utf-8"))
print(f"recognizer saw {pred['ner_seen']:,} of them during its own training")

22,059 test sentences, 4 conditions
recognizer saw 17,653 of them during its own training


## Cell 3: Entities in the English references

Same model as the projection pipeline, so the entity inventory matches the one
the corpus was built from.

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger"])

KEEP = {"PERSON","ORG","GPE","LOC","NORP","FAC","LAW","EVENT",
        "PRODUCT","WORK_OF_ART","LANGUAGE"}

ref_ents = []
for i, doc in enumerate(nlp.pipe(refs, batch_size=256)):
    ref_ents.append([(e.text.strip(), e.label_) for e in doc.ents
                     if e.label_ in KEEP and e.text.strip()])
    if i % 5000 == 0:
        print(f"  {i:,}/{N:,}", flush=True)

total = sum(len(e) for e in ref_ents)
with_ent = sum(1 for e in ref_ents if e)
print(f"\nreference entities   : {total:,}")
print(f"sentences with any   : {with_ent:,} ({with_ent/N*100:.1f}%)")
print(f"\n{'Type':<14}{'count':>8}")
for lab, n in Counter(l for e in ref_ents for _, l in e).most_common():
    print(f"{lab:<14}{n:>8,}")

  0/22,059
  5,000/22,059
  10,000/22,059
  15,000/22,059
  20,000/22,059

reference entities   : 31,280
sentences with any   : 22,059 (100.0%)

Type             count
PERSON          11,490
ORG              8,489
GPE              8,435
NORP             1,310
LOC                430
PRODUCT            288
LANGUAGE           264
WORK_OF_ART        243
FAC                219
EVENT               79
LAW                 33


## Cell 4: Presence test

An entity counts as rendered if its surface form appears in the hypothesis at
word boundaries, ignoring case and punctuation. This is deliberately strict on
identity and lenient on formatting: `Aizawl's` counts, `Aizawlian` does not.

In [5]:
def norm(s):
    s = unicodedata.normalize("NFKC", s).casefold()
    s = re.sub(r"[^\w\s]", " ", s)
    return " ".join(s.split())

_cache = {}
def contains(hyp_norm, ent):
    e = norm(ent)
    if not e:
        return False
    pat = _cache.get(e)
    if pat is None:
        pat = _cache[e] = re.compile(r"(?<!\w)" + re.escape(e) + r"(?!\w)")
    return pat.search(hyp_norm) is not None

# self-test
checks = [("Liana took three weeks off.", "Liana", True),
          ("The Aizawlian dialect.", "Aizawl", False),
          ("They reached Aizawl, then home.", "Aizawl", True),
          ("Niharika Rai spoke.", "Pi Niharika Rai", False)]
for h, e, want in checks:
    assert contains(norm(h), e) == want, (h, e)
print("presence test verified")

presence test verified


## Cell 5: Entity retention by condition

In [6]:
hyp_norm = {c: [norm(h) for h in hyps[c]] for c in CONDS}

# flatten every reference entity into one list, remembering its sentence
flat = [(i, txt, lab) for i, ents in enumerate(ref_ents) for txt, lab in ents]
flat_sent = np.array([i for i, _, _ in flat])
flat_type = np.array([lab for _, _, lab in flat])
M = len(flat)

retained = {}
for c in CONDS:
    hn = hyp_norm[c]
    retained[c] = np.fromiter(
        (contains(hn[i], txt) for i, txt, _ in flat), dtype=bool, count=M)

assert all(len(v) == M for v in retained.values())
print(f"{'Condition':<26}{'retained':>11}{'rate':>9}{'delta':>9}")
print("-" * 56)
base = retained["baseline"].mean()
rates = {}
for c in CONDS:
    r = retained[c].mean(); rates[c] = r
    d = "" if c == "baseline" else f"{(r-base)*100:+8.2f}"
    print(f"{LABEL[c]:<26}{int(retained[c].sum()):>11,}{r*100:>8.2f}%{d:>9}")
print(f"\ntotal reference entities: {M:,}")

Condition                    retained     rate    delta
--------------------------------------------------------
no markup                      30,214   96.59%         
oracle (projection)            30,472   97.42%    +0.82
predicted, stem-aligned        30,059   96.10%    -0.50
predicted, whole-token         29,808   95.29%    -1.30

total reference entities: 31,280


## Cell 6: By entity type

In [7]:
types = [t for t, _ in Counter(flat_type).most_common()]
print(f"{'Type':<14}{'n':>7}{'base':>9}{'oracle':>9}{'pred':>9}"
      f"{'or-base':>9}{'pr-base':>9}")
print("-" * 66)
per_type = {}
for t in types:
    m = flat_type == t
    n = int(m.sum())
    b  = retained["baseline"][m].mean()
    o  = retained["marked"][m].mean()
    p_ = retained["predicted"][m].mean()
    per_type[t] = {"n": n, "baseline": b, "marked": o, "predicted": p_}
    print(f"{t:<14}{n:>7,}{b*100:>8.1f}%{o*100:>8.1f}%{p_*100:>8.1f}%"
          f"{(o-b)*100:>+9.2f}{(p_-b)*100:>+9.2f}")

Type                n     base   oracle     pred  or-base  pr-base
------------------------------------------------------------------
PERSON         11,490    96.7%    97.6%    95.1%    +0.91    -1.60
ORG             8,489    96.0%    96.9%    96.3%    +0.91    +0.27
GPE             8,435    98.5%    98.9%    98.5%    +0.40    +0.02
NORP            1,310    92.4%    94.2%    92.8%    +1.83    +0.46
LOC               430    91.6%    93.3%    93.0%    +1.63    +1.40
PRODUCT           288    95.1%    95.8%    93.4%    +0.69    -1.74
LANGUAGE          264    99.6%   100.0%    99.6%    +0.38    +0.00
WORK_OF_ART       243    89.3%    90.9%    90.9%    +1.65    +1.65
FAC               219    88.6%    90.9%    85.8%    +2.28    -2.74
EVENT              79    87.3%    87.3%    86.1%    +0.00    -1.27
LAW                33    90.9%    90.9%    90.9%    +0.00    +0.00


## Cell 7: Significance

Retention is a paired binary outcome per entity, so McNemar's test on the
discordant pairs is the right instrument: it counts entities one system renders
and the other does not.

In [8]:
from scipy.stats import binomtest

def mcnemar(a, b, name):
    only_a = int((a & ~b).sum())
    only_b = int((b & ~a).sum())
    n = only_a + only_b
    if n == 0:
        print(f"{name:<34} no discordant pairs")
        return None
    res = binomtest(only_a, n, 0.5)
    print(f"{name:<34}{only_a:>7,}{only_b:>8,}{only_a-only_b:>+9,}"
          f"{res.pvalue:>12.2e}")
    return {"only_a": only_a, "only_b": only_b, "p": float(res.pvalue)}

print(f"{'Comparison':<34}{'A only':>7}{'B only':>8}{'net':>9}{'p':>12}")
print("-" * 70)
sig = {}
sig["oracle vs baseline"]    = mcnemar(retained["marked"], retained["baseline"],
                                       "oracle (A) vs baseline (B)")
sig["predicted vs baseline"] = mcnemar(retained["predicted"], retained["baseline"],
                                       "predicted (A) vs baseline (B)")
sig["oracle vs predicted"]   = mcnemar(retained["marked"], retained["predicted"],
                                       "oracle (A) vs predicted (B)")
sig["raw vs stem-aligned"]   = mcnemar(retained["predicted_raw"], retained["predicted"],
                                       "whole-token (A) vs aligned (B)")

Comparison                         A only  B only      net           p
----------------------------------------------------------------------
oracle (A) vs baseline (B)            410     152     +258    2.34e-28
predicted (A) vs baseline (B)         322     477     -155    4.64e-08
oracle (A) vs predicted (B)           445      32     +413    3.74e-94
whole-token (A) vs aligned (B)         43     294     -251    4.10e-47


## Cell 8: Save and emit LaTeX

In [9]:
out = {
    "reference_entities": int(M),
    "sentences": N,
    "retention": {c: round(float(rates[c]) * 100, 2) for c in CONDS},
    "retention_delta_vs_baseline": {
        c: round(float(rates[c] - rates["baseline"]) * 100, 2)
        for c in CONDS if c != "baseline"},
    "per_type": {t: {"n": v["n"],
                     "baseline": round(v["baseline"]*100, 2),
                     "marked": round(v["marked"]*100, 2),
                     "predicted": round(v["predicted"]*100, 2)}
                 for t, v in per_type.items()},
    "mcnemar": sig,
}
write_json(out, RES / "entity_fidelity_v2.json", indent=2)
print(f"-> {(RES/'entity_fidelity_v2.json').relative_to(ROOT)}\n")

print("% ---- Table: entity retention ----")
for c in CONDS:
    d = "---" if c == "baseline" else f"${(rates[c]-rates['baseline'])*100:+.2f}$"
    print(f"{LABEL[c]:<26}& {int(retained[c].sum()):,} & {rates[c]*100:.2f} & {d} \\\\")

print("\n% ---- Table: retention by type ----")
for t in types:
    v = per_type[t]
    esc = t.replace("_", "\\_")
    print(f"{esc:<14}& {v['n']:,} & {v['baseline']*100:.1f} & {v['marked']*100:.1f} "
          f"& {v['predicted']*100:.1f} & ${(v['marked']-v['baseline'])*100:+.2f}$ "
          f"& ${(v['predicted']-v['baseline'])*100:+.2f}$ \\\\")

-> results\mt\entity_fidelity_v2.json

% ---- Table: entity retention ----
no markup                 & 30,214 & 96.59 & --- \\
oracle (projection)       & 30,472 & 97.42 & $+0.82$ \\
predicted, stem-aligned   & 30,059 & 96.10 & $-0.50$ \\
predicted, whole-token    & 29,808 & 95.29 & $-1.30$ \\

% ---- Table: retention by type ----
PERSON        & 11,490 & 96.7 & 97.6 & 95.1 & $+0.91$ & $-1.60$ \\
ORG           & 8,489 & 96.0 & 96.9 & 96.3 & $+0.91$ & $+0.27$ \\
GPE           & 8,435 & 98.5 & 98.9 & 98.5 & $+0.40$ & $+0.02$ \\
NORP          & 1,310 & 92.4 & 94.2 & 92.8 & $+1.83$ & $+0.46$ \\
LOC           & 430 & 91.6 & 93.3 & 93.0 & $+1.63$ & $+1.40$ \\
PRODUCT       & 288 & 95.1 & 95.8 & 93.4 & $+0.69$ & $-1.74$ \\
LANGUAGE      & 264 & 99.6 & 100.0 & 99.6 & $+0.38$ & $+0.00$ \\
WORK\_OF\_ART & 243 & 89.3 & 90.9 & 90.9 & $+1.65$ & $+1.65$ \\
FAC           & 219 & 88.6 & 90.9 & 85.8 & $+2.28$ & $-2.74$ \\
EVENT         & 79 & 87.3 & 87.3 & 86.1 & $+0.00$ & $-1.27$ \\
LAW           & 33